# Phone brand ratings: correspondence analysis of means

**Correspondence Analysis** · a worked example from the StatNex teaching guide

**The story.** A review site averaged its readers' ratings (1-5) of four phone brands on five attributes. Marketing wants a picture of where each brand stands.

**The question.** Which attributes is each brand known for?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/correspondence_analysis__means.csv` (5 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Attribute,Nova,Pixelon,Orbit,Zentra
Design,4.31,3.62,3.88,3.17
Battery life,3.12,4.41,3.55,3.93
Camera,4.52,3.38,4.09,2.98
Value for money,2.64,3.79,3.31,4.47
Durability,3.46,4.02,3.61,4.18

These cells save `totals.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile totals.py
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
# A table of means: one row per item, one column per object rated.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Nova", "Pixelon", "Orbit", "Zentra"]
table = data.set_index("Attribute")[columns].apply(pd.to_numeric, errors="coerce")
# A row or column named like a total (Total, Sum, or a Japanese word for a total, as listed in totals.py) is a margin, not data,
# so it is left out, as the app does. Spaces, underscores, case and full-width letters
# are ignored when names are compared.
from totals import is_total

table = table.loc[[not is_total(row) for row in table.index],
                  [not is_total(column) for column in table.columns]]
# A row with a blank cell is left out.
table = table.dropna()
counts = table.to_numpy(float)
n_rows, n_columns = counts.shape
row_labels, column_labels = list(table.index), list(table.columns)

## The table scaled to proportions

In [ ]:
# The table is divided by its grand total, so it sums to 1.
P = counts / counts.sum()
row_weight = P.sum(axis=1)
row_margin, column_margin = P.sum(axis=1), P.sum(axis=0)
row_mass, column_mass = row_margin / P.sum(), column_margin / P.sum()
column_weight = column_mass

## Cross tabulation

In [ ]:
# The margins of a table of means are means.
grid = np.column_stack([counts, counts.mean(axis=1)])
grid = np.vstack([grid, [*counts.mean(axis=0), counts.mean()]])
crosstab = pd.DataFrame(grid, index=[*row_labels, "Overall"], columns=[*column_labels, "Overall"])
print(crosstab)

## Decomposition

In [ ]:
# The standardised residuals from independence, (p_ij - r_i c_j) / sqrt(r_i c_j), are
# split by a singular value decomposition. A table with R rows and C columns has
# min(R, C) - 1 axes; an axis with no inertia at all is dropped.
expected = np.outer(row_margin, column_margin) / P.sum()
residual = (P - expected) / np.sqrt(expected)
U, singular, Vt = np.linalg.svd(residual, full_matrices=False)
keep = min(n_rows, n_columns) - 1
U, singular, V = U[:, :keep], singular[:keep], Vt[:keep].T
live = singular > 1e-12 * max(1.0, singular[0])
U, singular, V = U[:, live], singular[live], V[:, live]
# An eigenvector and its negative describe the same axis, so its direction is a
# convention. The app's: flip each axis so its largest row-side component (in absolute
# value) is positive. Another program may draw the map mirrored, and when two
# components tie for the largest (two rows of equal weight, say) rounding decides.
for k in range(len(singular)):
    if U[np.argmax(np.abs(U[:, k])), k] < 0:
        U[:, k], V[:, k] = -U[:, k], -V[:, k]
eigen = singular**2
share = eigen / eigen.sum()
axes = [f"Axis {k + 1}" for k in range(len(singular))]

## Eigenvalues

In [ ]:
# The eigenvalue (the singular value squared) is the inertia the axis carries; its share
# of the total decides how many axes are worth reading.
eigenvalues = pd.DataFrame({"singular value": singular, "eigenvalue": eigen, "share": share,
                            "cumulative share": share.cumsum()}, index=axes)
print()
print(eigenvalues)

## Row and column summaries

In [ ]:
# For each level and each axis:
#   eigenvector                    the singular vector itself
#   score                          eigenvector / sqrt(weight) * sqrt(singular value): the
#                                  symmetric scaling that puts rows and columns on one map
#   contribution to the axis       eigenvector^2: how much of the axis the level makes
#   contribution within the level  the share of the level's inertia the axis explains
def summary(labels, vectors, scores, mass):
    parts = vectors**2 * eigen
    level_totals = parts.sum(axis=1, keepdims=True)
    # A level with no inertia (a profile equal to the average one, such as a Total row)
    # gets 0 rather than 0 / 0. Its inertia is zero only up to rounding, so "zero" means
    # below 1e-12 of the total.
    zero = level_totals <= 1e-12 * eigen.sum()
    within_level = np.divide(parts, level_totals, out=np.zeros_like(parts), where=~zero)
    groups = {"Eigenvector": vectors, "Score": scores, "Contribution to the axis": vectors**2,
              "Contribution within the level": within_level}
    out = pd.DataFrame(np.hstack(list(groups.values())), index=labels,
                       columns=[f"{group} {axis}" for group in groups for axis in axes])
    return out


row_scores = U / np.sqrt(row_weight)[:, None] * np.sqrt(singular)
column_scores = V / np.sqrt(column_weight)[:, None] * np.sqrt(singular)
row_summary = summary(row_labels, U, row_scores, row_mass)
column_summary = summary(column_labels, V, column_scores, column_mass)
print()
print(row_summary.round(4).T)
print()
print(column_summary.round(4).T)

## Scores

In [ ]:
# The coordinates of the map: rows and columns on the same axes. A row and a column that
# lie in the same direction from the centre occur together more often than independence
# would predict; the exact distance between a row point and a column point means little.
scores = pd.DataFrame(np.vstack([row_scores, column_scores]), index=[*row_labels, *column_labels],
                      columns=axes)
print()
print(scores.round(4))

## What the result says

Here the table holds mean ratings, not counts. Correspondence analysis turns it into a map: a brand and an attribute with similar scores on an axis go together.

Axis 1 has a share of 94.1%, so it shows most of the pattern. On it, Nova (0.51) sits with Camera (0.49) and Design (0.35), and Zentra (-0.45) sits with Value for money (-0.47). Axis 2 picks out Pixelon (0.28) with Battery life (0.32). Orbit has small scores on both axes (0.17 and -0.04), so it is known for nothing in particular.

Each brand has a recognisable profile. But the differences are small: every mean rating lies between 2.64 and 4.52, and the map makes small gaps look large.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.